Benchmark IMM dense image-matching models (MatchAnything-ELOFTR/ROMA, ELOFTR, ROMA, etc.) on a synthetic scientific panel-reuse pair dataset (full duplicate, full/partial overlap, no-match) using mask-aware metrics, runtime profiling, and a resumable JSONL-based pipeline that generates per-model reports and a leaderboard as each model finishes.

## Cell 1 — Install IMM (clone + editable install)

IMM recommends cloning --recursive and pip install -e ..

link: https://github.com/gmberton/image-matching-models

In [1]:
# !rm -rf /kaggle/working/image-matching-models
# %cd /kaggle/working

# # Force git to use https instead of ssh for GitHub (avoids the yes/no prompt)
# !git config --global url."https://github.com/".insteadOf git@github.com:
# !git config --global url."https://github.com/".insteadOf git://github.com/

# # Clone + submodules with live output
# !git clone https://github.com/gmberton/image-matching-models /kaggle/working/image-matching-models
# %cd /kaggle/working/image-matching-models
# !git submodule sync --recursive
# !git submodule update --init --recursive --progress

# !pip install -e . -q

In [ ]:
!pip install git+https://github.com/alexstoken/image-matching-models.git

  Cloning https://github.com/alexstoken/image-matching-models.git to /tmp/pip-req-build-horwhtyd
  Running command git clone --filter=blob:none --quiet https://github.com/alexstoken/image-matching-models.git /tmp/pip-req-build-horwhtyd
  Resolved https://github.com/alexstoken/image-matching-models.git to commit ed6ed320d26bcda10eab4cd36de7f571ebaa0f83
  Running command git submodule update --init --recursive -q
The authenticity of host 'github.com (140.82.116.4)' can't be established.
ED25519 key fingerprint is SHA256:+DiY3wvvV6TuJJhbpZisF/zLDA0zPMSvHdkr4UvCOqU.
This key is not known by any other names
Are you sure you want to continue connecting (yes/no/[fingerprint])? 

In [6]:
# testing
from matching import get_matcher
from matching.viz import plot_matches

## Cell 2 — Write the resumable benchmark script

In [9]:
%%writefile bench_imm_pairs.py
#!/usr/bin/env python3
"""
bench_imm_pairs.py

Resumable benchmarking of IMM matchers on a (images/, masks/) pair dataset.

Dataset layout expected (your layout):
  <root>/
    images/<category>/batch_*/<pair_folder>/<idx>_a.png, <idx>_b.png
    masks/<category>/batch_*/<pair_folder>/<idx>_a.png, <idx>_b.png
    meta/<category>/batch_*/<pair_folder>/<idx>.json   (optional)
"""

import os
import json
import time
import math
import traceback
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import cv2
from tqdm import tqdm

# IMM
from matching import get_matcher


POS_CATEGORIES_DEFAULT = ["full_duplicate", "full_overlap_crop", "partial_overlap"]
NEG_CATEGORIES_DEFAULT = ["no_match"]


def read_mask_grayscale(path: Path) -> Optional[np.ndarray]:
    if not path.exists():
        return None
    m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return m


def resize_mask_to(mask: np.ndarray, w: int, h: int) -> np.ndarray:
    # nearest-neighbor to preserve labels
    return cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)


def pick_kpts(result: Dict[str, Any], prefer_inliers: bool = True) -> Tuple[np.ndarray, np.ndarray]:
    """
    IMM example shows keys:
      matched_kpts0, matched_kpts1, inlier_kpts0, inlier_kpts1, num_inliers, H, ...
    We'll prefer inliers if present and non-empty.
    """
    def _as_np(x):
        if x is None:
            return None
        if isinstance(x, np.ndarray):
            return x
        try:
            import torch
            if isinstance(x, torch.Tensor):
                return x.detach().cpu().numpy()
        except Exception:
            pass
        return np.asarray(x)

    mk0 = _as_np(result.get("matched_kpts0"))
    mk1 = _as_np(result.get("matched_kpts1"))
    ik0 = _as_np(result.get("inlier_kpts0"))
    ik1 = _as_np(result.get("inlier_kpts1"))

    if prefer_inliers and ik0 is not None and ik1 is not None and len(ik0) > 0 and len(ik1) > 0:
        return ik0, ik1
    if mk0 is not None and mk1 is not None:
        return mk0, mk1

    # fallback: empty
    return np.zeros((0, 2), dtype=np.float32), np.zeros((0, 2), dtype=np.float32)


def rasterize_points(points_xy: np.ndarray, w: int, h: int, radius: int = 3) -> np.ndarray:
    """
    Convert Nx2 float xy points into a binary hit-map with small dilation.
    """
    hit = np.zeros((h, w), dtype=np.uint8)
    if points_xy.size == 0:
        return hit
    xs = np.clip(np.round(points_xy[:, 0]).astype(int), 0, w - 1)
    ys = np.clip(np.round(points_xy[:, 1]).astype(int), 0, h - 1)
    hit[ys, xs] = 1
    if radius > 0:
        k = 2 * radius + 1
        kernel = np.ones((k, k), np.uint8)
        hit = cv2.dilate(hit, kernel, iterations=1)
    return hit


def safe_div(a: float, b: float) -> float:
    return float(a) / float(b) if b not in (0, 0.0) else float("nan")


def enumerate_samples(data_root: Path, categories: Optional[List[str]] = None) -> List[Dict[str, Any]]:
    images_root = data_root / "images"
    masks_root = data_root / "masks"
    meta_root = data_root / "meta"

    if categories is None:
        categories = sorted([d.name for d in images_root.iterdir() if d.is_dir()])

    samples: List[Dict[str, Any]] = []
    for cat in categories:
        cat_dir = images_root / cat
        if not cat_dir.exists():
            continue
        for batch_dir in sorted([d for d in cat_dir.iterdir() if d.is_dir() and d.name.startswith("batch_")]):
            for pair_dir in sorted([d for d in batch_dir.iterdir() if d.is_dir()]):
                # idx_a.png pattern
                for a_path in sorted(pair_dir.glob("*_a.png")):
                    idx = a_path.stem[:-2]  # remove "_a"
                    b_path = pair_dir / f"{idx}_b.png"
                    if not b_path.exists():
                        continue

                    m_a = masks_root / cat / batch_dir.name / pair_dir.name / f"{idx}_a.png"
                    m_b = masks_root / cat / batch_dir.name / pair_dir.name / f"{idx}_b.png"
                    meta_p = meta_root / cat / batch_dir.name / pair_dir.name / f"{idx}.json"

                    sample_id = f"{cat}/{batch_dir.name}/{pair_dir.name}/{idx}"
                    samples.append({
                        "sample_id": sample_id,
                        "category": cat,
                        "batch": batch_dir.name,
                        "pair_folder": pair_dir.name,
                        "idx": idx,
                        "img_a": str(a_path),
                        "img_b": str(b_path),
                        "mask_a": str(m_a),
                        "mask_b": str(m_b),
                        "meta": str(meta_p),
                    })

    return samples


def load_done_ids(jsonl_path: Path) -> set:
    done = set()
    if not jsonl_path.exists():
        return done
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                j = json.loads(line)
                sid = j.get("sample_id")
                if sid:
                    done.add(sid)
            except Exception:
                continue
    return done


def write_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def summarize_model(results_jsonl: Path, out_dir: Path, pos_cats: List[str], neg_cats: List[str]) -> Dict[str, Any]:
    # lightweight summary without pandas dependency
    rows = []
    with results_jsonl.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    ok = [r for r in rows if r.get("status") == "ok"]
    if not ok:
        return {"n_total": len(rows), "n_ok": 0}

    # category aggregates
    by_cat: Dict[str, List[Dict[str, Any]]] = {}
    for r in ok:
        by_cat.setdefault(r["category"], []).append(r)

    def mean(xs):
        xs = [x for x in xs if x is not None and not (isinstance(x, float) and math.isnan(x))]
        return float(np.mean(xs)) if xs else float("nan")

    summary = {
        "n_total": len(rows),
        "n_ok": len(ok),
        "n_err": len(rows) - len(ok),
        "time_sec_mean": mean([r.get("dt_match_sec") for r in ok]),
        "time_sec_median": float(np.median([r.get("dt_match_sec") for r in ok])),
        "n_matches_mean": mean([r.get("n_matches") for r in ok]),
        "precision_in_mask_mean": mean([r.get("precision_in_mask") for r in ok]),
        "coverage_mean": mean([r.get("coverage_mean") for r in ok]),
        "per_category": {}
    }

    for cat, rs in by_cat.items():
        summary["per_category"][cat] = {
            "n": len(rs),
            "time_sec_mean": mean([r.get("dt_match_sec") for r in rs]),
            "n_matches_mean": mean([r.get("n_matches") for r in rs]),
            "precision_in_mask_mean": mean([r.get("precision_in_mask") for r in rs]),
            "coverage_mean": mean([r.get("coverage_mean") for r in rs]),
        }

    # detection metrics (needs sklearn if available)
    try:
        from sklearn.metrics import roc_auc_score, average_precision_score

        labels = []
        scores = []
        for r in ok:
            cat = r["category"]
            if cat in pos_cats:
                labels.append(1)
            elif cat in neg_cats:
                labels.append(0)
            else:
                continue
            scores.append(float(r.get("score", 0.0)))

        if len(set(labels)) == 2:
            summary["roc_auc"] = float(roc_auc_score(labels, scores))
            summary["avg_precision"] = float(average_precision_score(labels, scores))
        else:
            summary["roc_auc"] = float("nan")
            summary["avg_precision"] = float("nan")
    except Exception:
        summary["roc_auc"] = float("nan")
        summary["avg_precision"] = float("nan")

    # write summary.json + report.md
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    # simple markdown report
    lines = []
    lines.append(f"# Report: {out_dir.name}")
    lines.append("")
    lines.append(f"- total: **{summary['n_total']}**, ok: **{summary['n_ok']}**, err: **{summary['n_err']}**")
    lines.append(f"- match time mean/median (sec): **{summary['time_sec_mean']:.4f}** / **{summary['time_sec_median']:.4f}**")
    lines.append(f"- ROC-AUC (match vs no_match): **{summary.get('roc_auc', float('nan')):.4f}**")
    lines.append(f"- Avg Precision (PR-AUC): **{summary.get('avg_precision', float('nan')):.4f}**")
    lines.append("")
    lines.append("## Per-category")
    lines.append("")
    lines.append("| category | n | time_mean_s | n_matches_mean | precision_in_mask_mean | coverage_mean |")
    lines.append("|---|---:|---:|---:|---:|---:|")
    for cat, s in summary["per_category"].items():
        lines.append(
            f"| {cat} | {s['n']} | {s['time_sec_mean']:.4f} | {s['n_matches_mean']:.2f} | "
            f"{s['precision_in_mask_mean']:.4f} | {s['coverage_mean']:.4f} |"
        )

    (out_dir / "report.md").write_text("\n".join(lines), encoding="utf-8")
    return summary


def update_leaderboard(out_root: Path) -> None:
    # collect all summaries
    rows = []
    for model_dir in sorted([d for d in out_root.iterdir() if d.is_dir()]):
        s = model_dir / "summary.json"
        if not s.exists():
            continue
        try:
            j = json.loads(s.read_text(encoding="utf-8"))
            rows.append({
                "model": model_dir.name,
                "n_ok": j.get("n_ok"),
                "time_mean_s": j.get("time_sec_mean"),
                "roc_auc": j.get("roc_auc"),
                "avg_precision": j.get("avg_precision"),
                "precision_in_mask_mean": j.get("precision_in_mask_mean"),
                "coverage_mean": j.get("coverage_mean"),
            })
        except Exception:
            continue

    # write csv + md (no pandas needed)
    if not rows:
        return

    # sort: best avg_precision then roc_auc then speed
    rows_sorted = sorted(
        rows,
        key=lambda r: (
            -(r["avg_precision"] if r["avg_precision"] == r["avg_precision"] else -1e9),
            -(r["roc_auc"] if r["roc_auc"] == r["roc_auc"] else -1e9),
            (r["time_mean_s"] if r["time_mean_s"] == r["time_mean_s"] else 1e9),
        )
    )

    csv_path = out_root / "leaderboard.csv"
    md_path = out_root / "leaderboard.md"

    # CSV
    header = list(rows_sorted[0].keys())
    with csv_path.open("w", encoding="utf-8") as f:
        f.write(",".join(header) + "\n")
        for r in rows_sorted:
            f.write(",".join("" if r[h] is None else str(r[h]) for h in header) + "\n")

    # Markdown
    lines = ["# Leaderboard", "", "| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
    for r in rows_sorted:
        fmt = []
        for h in header:
            v = r[h]
            if isinstance(v, float):
                fmt.append(f"{v:.4f}" if abs(v) < 1e6 else str(v))
            else:
                fmt.append("" if v is None else str(v))
        lines.append("| " + " | ".join(fmt) + " |")
    md_path.write_text("\n".join(lines), encoding="utf-8")


def main():
    import argparse

    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", type=str, required=True)
    ap.add_argument("--out_root", type=str, required=True)
    ap.add_argument("--models", type=str, required=True, help="Comma-separated IMM matcher names")
    ap.add_argument("--device", type=str, default="cuda")
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--im_size", type=int, default=832)
    ap.add_argument("--prefer_inliers", action="store_true")
    ap.add_argument("--radius", type=int, default=3, help="dilation radius for coverage metric")
    ap.add_argument("--max_per_category", type=int, default=0, help="0 = no limit")
    ap.add_argument("--pos_cats", type=str, default=",".join(POS_CATEGORIES_DEFAULT))
    ap.add_argument("--neg_cats", type=str, default=",".join(NEG_CATEGORIES_DEFAULT))
    args = ap.parse_args()

    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)

    data_root = Path(args.data_root)
    out_root = Path(args.out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    models = [m.strip() for m in args.models.split(",") if m.strip()]
    pos_cats = [c.strip() for c in args.pos_cats.split(",") if c.strip()]
    neg_cats = [c.strip() for c in args.neg_cats.split(",") if c.strip()]

    all_samples = enumerate_samples(data_root)
    # optional sampling per category (useful for your “sample data first”)
    if args.max_per_category and args.max_per_category > 0:
        by = {}
        for s in all_samples:
            by.setdefault(s["category"], []).append(s)
        kept = []
        for cat, ss in by.items():
            kept.extend(ss[: args.max_per_category])
        all_samples = kept

    print(f"[INFO] Found {len(all_samples)} samples")

    for model_name in models:
        model_dir = out_root / model_name
        results_jsonl = model_dir / "predictions.jsonl"
        errors_jsonl = model_dir / "errors.jsonl"
        done_ids = load_done_ids(results_jsonl)
        print(f"\n=== Model: {model_name} | done: {len(done_ids)} / {len(all_samples)} ===")

        # load matcher
        try:
            matcher = get_matcher(model_name, device=args.device)
        except Exception as e:
            print(f"[ERROR] Failed to load matcher {model_name}: {e}")
            write_jsonl(errors_jsonl, {
                "ts": time.time(),
                "model": model_name,
                "status": "load_error",
                "error": str(e),
                "traceback": traceback.format_exc(),
            })
            continue

        pbar = tqdm(all_samples, desc=f"{model_name}", unit="pair")
        for s in pbar:
            sid = s["sample_id"]
            if sid in done_ids:
                continue

            cat = s["category"]
            label = 1 if cat in pos_cats else 0 if cat in neg_cats else None

            try:
                # load images through IMM helper (lets each matcher do correct preprocessing)
                img0 = matcher.load_image(s["img_a"], resize=args.im_size)
                img1 = matcher.load_image(s["img_b"], resize=args.im_size)

                # get processed sizes to resize masks consistently
                # works for torch tensors (C,H,W) or (1,C,H,W)
                shp0 = img0.shape
                shp1 = img1.shape
                h0, w0 = int(shp0[-2]), int(shp0[-1])
                h1, w1 = int(shp1[-2]), int(shp1[-1])

                m0 = read_mask_grayscale(Path(s["mask_a"]))
                m1 = read_mask_grayscale(Path(s["mask_b"]))
                if m0 is None or m1 is None:
                    # still run (you might have missing masks in some cases)
                    m0r = np.zeros((h0, w0), dtype=np.uint8)
                    m1r = np.zeros((h1, w1), dtype=np.uint8)
                else:
                    m0r = resize_mask_to(m0, w0, h0)
                    m1r = resize_mask_to(m1, w1, h1)

                t0 = time.perf_counter()
                result = matcher(img0, img1)
                dt = time.perf_counter() - t0

                k0, k1 = pick_kpts(result, prefer_inliers=args.prefer_inliers)
                n_matches = int(len(k0))

                # keypoint-in-mask counting
                if n_matches > 0:
                    x0 = np.clip(np.round(k0[:, 0]).astype(int), 0, w0 - 1)
                    y0 = np.clip(np.round(k0[:, 1]).astype(int), 0, h0 - 1)
                    x1 = np.clip(np.round(k1[:, 0]).astype(int), 0, w1 - 1)
                    y1 = np.clip(np.round(k1[:, 1]).astype(int), 0, h1 - 1)
                    inside = (m0r[y0, x0] > 0) & (m1r[y1, x1] > 0)
                    n_pos = int(np.sum(inside))
                else:
                    n_pos = 0

                precision_in_mask = safe_div(n_pos, n_matches) if n_matches else 0.0

                # coverage (recall-like) on each side
                hit0 = rasterize_points(k0, w0, h0, radius=args.radius)
                hit1 = rasterize_points(k1, w1, h1, radius=args.radius)
                ma = (m0r > 0).astype(np.uint8)
                mb = (m1r > 0).astype(np.uint8)
                area_a = int(ma.sum())
                area_b = int(mb.sum())
                cov_a = safe_div(int((hit0 & ma).sum()), area_a) if area_a > 0 else float("nan")
                cov_b = safe_div(int((hit1 & mb).sum()), area_b) if area_b > 0 else float("nan")
                cov_mean = float(np.nanmean([cov_a, cov_b]))

                # scalar score for match/no_match detection
                # prefer RANSAC inliers if matcher provides it, else n_matches
                score = result.get("num_inliers", None)
                if score is None:
                    score = n_matches
                score = float(score)

                rec = {
                    "ts": time.time(),
                    "status": "ok",
                    "model": model_name,
                    "sample_id": sid,
                    "category": cat,
                    "label": label,
                    "img_a": s["img_a"],
                    "img_b": s["img_b"],
                    "n_matches": n_matches,
                    "n_pos_matches": n_pos,
                    "precision_in_mask": float(precision_in_mask),
                    "coverage_a": float(cov_a) if cov_a == cov_a else None,
                    "coverage_b": float(cov_b) if cov_b == cov_b else None,
                    "coverage_mean": float(cov_mean) if cov_mean == cov_mean else None,
                    "dt_match_sec": float(dt),
                    "score": score,
                    "num_inliers": result.get("num_inliers", None),
                }
                write_jsonl(results_jsonl, rec)
                done_ids.add(sid)

                pbar.set_postfix({
                    "dt(s)": f"{dt:.3f}",
                    "m": n_matches,
                    "p_in": f"{precision_in_mask:.2f}",
                })

            except Exception as e:
                write_jsonl(errors_jsonl, {
                    "ts": time.time(),
                    "model": model_name,
                    "sample_id": sid,
                    "category": cat,
                    "status": "error",
                    "error": str(e),
                    "traceback": traceback.format_exc(),
                })
                continue

        # report immediately after the model finishes
        print(f"[INFO] Summarizing {model_name} ...")
        summary = summarize_model(results_jsonl, model_dir, pos_cats=pos_cats, neg_cats=neg_cats)
        print(f"[INFO] Wrote: {model_dir/'summary.json'} and {model_dir/'report.md'}")

        update_leaderboard(out_root)
        print(f"[INFO] Updated leaderboard at: {out_root/'leaderboard.md'}")


if __name__ == "__main__":
    main()

Overwriting bench_imm_pairs.py


## Cell 3 — Run benchmark on your sample dataset

Set DATA_ROOT to wherever you put pairs_v1 in Kaggle (as a Kaggle Dataset: /kaggle/input/...), or if you uploaded it into the notebook working dir, point accordingly.

In [11]:
import os, textwrap, subprocess, json
from pathlib import Path

DATA_ROOT = "/kaggle/input/panel-reuse-det-synth-pairs-v1/pairs_v1"   # <-- change this
OUT_ROOT  = "/kaggle/working/bench_out"

MODELS = [
    "matchanything-eloftr",
    "matchanything-roma",
    "eloftr",
    "roma",
    "tiny-roma",      # optional speed baseline
    "xfeat-star",     # optional strong semi-dense
]

# Start small on sample runs:
MAX_PER_CATEGORY = 50   # 0 = no limit

cmd = [
    "python", "bench_imm_pairs.py",
    "--data_root", DATA_ROOT,
    "--out_root", OUT_ROOT,
    "--models", ",".join(MODELS),
    "--device", "cuda",
    "--gpu", "0",
    "--im_size", "832",          # MatchAnything notes 832px default; also good for others :contentReference[oaicite:4]{index=4}
    "--prefer_inliers",          # uses inlier_kpts if available
    "--max_per_category", str(MAX_PER_CATEGORY),
]

print(" ".join(cmd))
subprocess.run(cmd, check=False)


python bench_imm_pairs.py --data_root /kaggle/input/panel-reuse-det-synth-pairs-v1/pairs_v1 --out_root /kaggle/working/bench_out --models matchanything-eloftr,matchanything-roma,eloftr,roma,tiny-roma,xfeat-star --device cuda --gpu 0 --im_size 832 --prefer_inliers --max_per_category 50


/kaggle/working/image-matching-models/matching/third_party/MatchAnything/imcui/third_party/MatchAnything/src/loftr/loftr_module/linear_attention.py:36: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/kaggle/working/image-matching-models/matching/third_party/MatchAnything/imcui/third_party/MatchAnything/src/loftr/loftr_module/linear_attention.py:74: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/kaggle/working/image-matching-models/matching/third_party/MatchAnything/imcui/third_party/MatchAnything/src/loftr/loftr_module/transformer_utils.py:40: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.

[INFO] Found 70 samples

=== Model: matchanything-eloftr | done: 0 / 70 ===


100%|██████████| 483M/483M [00:23<00:00, 20.5MB/s] 
2025-12-30 18:29:34.961 | INFO     | src.loftr.loftr_module.transformer:__init__:1441 - npe trainH,trainW,testH,testW: 832, 832, 832, 832


RepVGG Block, identity =  None
RepVGG Block, identity =  BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  None
RepVGG Block, identity =  BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  None
RepVGG Block, identity =  BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(256, eps=1e-05,

...

Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth


100%|██████████| 548M/548M [00:29<00:00, 19.4MB/s] 
matchanything-roma:   0%|          | 0/70 [00:00<?, ?pair/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
matchanything-roma:  51%|█████▏    | 36/70 [01:01<00:57,  1.68s/pair, dt(s)=1.635, m=4998, p_in=1.00]/kaggle/working/image-matching-models/bench_imm_pairs.py:444: RuntimeWarning: Mean of empty slice
  cov_mean = float(np.nanmean([cov_a, cov_b]))
matchanything-roma:  53%|█████▎    | 37/70 [01:03<00:55,  1.69s/pair, dt(s)=1.648, m=46, p_in=0.00]  /kaggle/working/image-matching-models/bench_imm_pairs.py:444: RuntimeWarning: Mean of empty slice
  cov_mean = float(np.nanmean([cov_a, cov_b]))
matchanything-roma:  54%|█████▍    | 38/70 [01:04<00:54,  1.69s/pair, dt(s)=1.

[INFO] Summarizing matchanything-roma ...
[INFO] Wrote: /kaggle/working/bench_out/matchanything-roma/summary.json and /kaggle/working/bench_out/matchanything-roma/report.md
[INFO] Updated leaderboard at: /kaggle/working/bench_out/leaderboard.md

=== Model: eloftr | done: 0 / 70 ===
[ERROR] Failed to load matcher eloftr: cannot import name 'full_default_cfg' from 'src.loftr' (/kaggle/working/image-matching-models/matching/third_party/MatchAnything/imcui/third_party/MatchAnything/src/loftr/__init__.py)

=== Model: roma | done: 0 / 70 ===
Downloading: "https://github.com/Parskatt/storage/releases/download/roma/roma_outdoor.pth" to /root/.cache/torch/hub/checkpoints/roma_outdoor.pth


 21%|██▏       | 90.5M/425M [00:11<00:43, 8.03MB/s]
Traceback (most recent call last):
  File "/kaggle/working/image-matching-models/bench_imm_pairs.py", line 503, in <module>
    main()
  File "/kaggle/working/image-matching-models/bench_imm_pairs.py", line 372, in main
    matcher = get_matcher(model_name, device=args.device)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/image-matching-models/matching/__init__.py", line 203, in get_matcher
    return roma.RomaMatcher(device, max_num_keypoints, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/image-matching-models/matching/im_models/roma.py", line 25, in __init__
    self.roma_model = roma_outdoor(device=device)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/image-matching-models/matching/third_party/RoMa/romatch/models/model_zoo/__init__.py", line 43, in roma_outdoor
    weights = torch.hub.load_state_dict_fr

KeyboardInterrupt: 

## Cell 4 — View leaderboard / open reports

In [12]:
from pathlib import Path

lb = Path("/kaggle/working/bench_out/leaderboard.md")
print(lb.read_text() if lb.exists() else "No leaderboard yet.")

# Show one model report:
rep = Path("/kaggle/working/bench_out/matchanything-eloftr/report.md")
print("\n" + rep.read_text() if rep.exists() else "No report yet.")

# Leaderboard

| model | n_ok | time_mean_s | roc_auc | avg_precision | precision_in_mask_mean | coverage_mean |
| --- | --- | --- | --- | --- | --- | --- |
| matchanything-roma | 70 | 1.6513 | 1.0000 | 1.0000 | 0.7416 | 0.4436 |
| matchanything-eloftr | 70 | 0.2445 | 0.7169 | 0.8959 | 0.4607 | 0.2074 |

# Report: matchanything-eloftr

- total: **70**, ok: **70**, err: **0**
- match time mean/median (sec): **0.2445** / **0.2345**
- ROC-AUC (match vs no_match): **0.7169**
- Avg Precision (PR-AUC): **0.8959**

## Per-category

| category | n | time_mean_s | n_matches_mean | precision_in_mask_mean | coverage_mean |
|---|---:|---:|---:|---:|---:|
| full_duplicate | 18 | 0.3106 | 7506.22 | 0.8889 | 0.3761 |
| full_overlap_crop | 18 | 0.2385 | 3019.89 | 0.8472 | 0.2188 |
| no_match | 18 | 0.2238 | 3.28 | 0.0000 | nan |
| partial_overlap | 16 | 0.2001 | 37.50 | 0.0625 | 0.0049 |
